# UFC round-by-round model (men's divisions)

Two models, multiplied together:

1. **Who wins**: P(fighter A wins). Every fight is entered from both sides, so the order the
   fighters are listed in (the winner comes first in about 60% of ufcstats.com bouts) can't leak.
2. **How it ends, given who wins**: P(round 1, 2, 3, 4, 5 or decision | that fighter wins).

`features.ufc.outcome_table` turns them into one table per fight whose cells add up to 1.
Every feature comes from the fighters' earlier fights only. Models are compared on every men's
fight since 1 January 2025, which they never see in training, against an Elo baseline.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from pycaret.classification import ClassificationExperiment

warnings.filterwarnings("ignore")

ROOT = Path.cwd() if (Path.cwd() / "features").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from features import ufc

DATA = ROOT / "UFC" / "datasets"
MODELS = ROOT / "UFC" / "models"
PREDICTIONS = ROOT / "UFC" / "predictions"
VERSION = 1
TRAIN_FROM, TEST_FROM = "2008-01-01", "2025-01-01"
MODEL_CHOICES = ["lr", "lightgbm", "catboost"]

In [2]:
fights = pd.read_csv(DATA / f"UFC_fights_v{VERSION}.csv", parse_dates=["Date"])
fighters = pd.read_csv(DATA / f"UFC_fighters_v{VERSION}.csv", parse_dates=["DOB"])
table = ufc.fight_table(fights, ufc.history_features(fights, fighters))
table = table[table["Date"] >= TRAIN_FROM].reset_index(drop=True)
print(f"{len(table)} men's fights since {TRAIN_FROM}; "
      f"{(table['Date'] < TEST_FROM).sum()} for training, {(table['Date'] >= TEST_FROM).sum()} held out")

7096 men's fights since 2008-01-01; 6276 for training, 820 held out


## 1. Who wins

In [3]:
winner_rows = ufc.winner_training_rows(table)
train_w = winner_rows[winner_rows["Date"] < TEST_FROM].drop(columns=["FightID", "Date"])
test_w = winner_rows[winner_rows["Date"] >= TEST_FROM].drop(columns=["FightID", "Date"])

winner_exp = ClassificationExperiment()
winner_exp.setup(data=train_w, test_data=test_w, target="AWon", session_id=123,
                 fold_strategy="timeseries", fold=5, data_split_shuffle=False, fold_shuffle=False,
                 categorical_features=["A_Stance", "B_Stance"], normalize=True, verbose=False, html=False)
winner_exp.add_metric("logloss", "Log Loss", log_loss, greater_is_better=False, target="pred_proba")
winner_model = winner_exp.compare_models(include=MODEL_CHOICES, sort="Log Loss", verbose=False)
print(winner_exp.pull()[["Accuracy", "AUC", "Log Loss"]])
print(winner_model)

# The app shows probabilities, so rescale them on the training folds (Platt scaling);
# the held-out fights are not used for this
winner_model = winner_exp.calibrate_model(winner_model, method="sigmoid", verbose=False)
print("calibrated:", winner_exp.pull().loc["Mean", ["Accuracy", "AUC", "Log Loss"]].to_dict())

          Accuracy     AUC  Log Loss
catboost    0.6108  0.6622    0.6488
lightgbm    0.6012  0.6452    0.6706
lr          0.5962  0.6309    0.6750


calibrated: {'Accuracy': 0.6138, 'AUC': 0.6684, 'Log Loss': 0.6452}


## 2. How the fight ends, given who wins

Trained on decided fights from the winner's side. How a fight ends is far noisier than who wins,
so this model only sees the inputs that bear on it: finishing and knockdown rates, durability,
pace, Elo, age and experience, plus the fight's context. Only 66 fights since 2008 ended in
round 4 or 5, so those two columns lean on the scheduled-rounds input; rounds a fight can't
reach are zeroed later.

In [4]:
def round_log_loss(y_true, y_pred, **kwargs):
    # PyCaret may pass the class names or their codes; codes follow the names' sorted order
    labels = sorted(ufc.OUTCOMES) if isinstance(np.asarray(y_true)[0], str) else list(range(len(ufc.OUTCOMES)))
    return log_loss(y_true, y_pred, labels=labels)

ROUND_INPUTS = ["KOWinRate", "SubWinRate", "KOLossRate", "SubLossRate", "DecisionRate", "AvgFightMin",
                "KDPer15", "KDAbsorbedPer15", "SubAttPer15", "SigLandedPerMin", "SigAbsorbedPerMin",
                "TDPer15", "Elo", "Fights", "Age"]
round_keep = [f"{side}_{c}" for c in ROUND_INPUTS for side in "ABD"] + ufc.CONTEXT

round_rows = ufc.round_training_rows(table)
train_r = round_rows[round_rows["Date"] < TEST_FROM].drop(columns=["FightID", "Date"])
test_r = round_rows[round_rows["Date"] >= TEST_FROM].drop(columns=["FightID", "Date"])
print(train_r["Outcome"].value_counts())

round_exp = ClassificationExperiment()
round_exp.setup(data=train_r, test_data=test_r, target="Outcome", session_id=123,
                fold_strategy="timeseries", fold=5, data_split_shuffle=False, fold_shuffle=False,
                ignore_features=[c for c in train_r.columns if c not in round_keep + ["Outcome"]],
                normalize=True, verbose=False, html=False)
round_exp.add_metric("logloss", "Log Loss", round_log_loss, greater_is_better=False, target="pred_proba")
round_model = round_exp.compare_models(include=MODEL_CHOICES, sort="Log Loss", verbose=False)
print(round_exp.pull()[["Accuracy", "Log Loss"]])
print(round_model)

Outcome
DEC    2873
R1     1644
R2     1018
R3      503
R4       37
R5       29
Name: count, dtype: int64


          Accuracy  Log Loss
lr          0.4785    1.2710
catboost    0.4751    1.3667
lightgbm    0.4688    1.5748
LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=1000,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=123, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)


## 3. Held-out check: every men's fight since 2025

Each fight's full table (both fighters, every round, decision) is scored by the probability it gave
to what actually happened. Baseline: Elo for who wins, times how often fights with that many
scheduled rounds ended each way in the training years.

In [5]:
SCORE_COLS = [f"prediction_score_{o}" for o in ufc.OUTCOMES]
FEATURE_COLS = [c for c in train_w.columns if c != "AWon"]

def win_probability(model, rows):
    p_ab = winner_exp.predict_model(model, data=rows[FEATURE_COLS], raw_score=True, verbose=False)["prediction_score_1"]
    p_ba = winner_exp.predict_model(model, data=ufc.swap(rows)[FEATURE_COLS], raw_score=True, verbose=False)["prediction_score_1"]
    return ufc.symmetric_win_probability(p_ab.to_numpy(), p_ba.to_numpy())

def how_it_ends(model, rows):
    return round_exp.predict_model(model, data=rows[FEATURE_COLS], raw_score=True, verbose=False)[SCORE_COLS].to_numpy()

test = table[(table["Date"] >= TEST_FROM) & table["Result"].isin(["W", "L"])].copy()
test["Outcome"] = [ufc.outcome_label(m, r) for m, r in zip(test["Method"], test["EndRound"])]
test = test[test["Outcome"].notna() & test["Rounds"].isin([3, 5])].reset_index(drop=True)
test["Winner"] = np.where(test["Result"] == "W", "A", "B")

shares = (round_rows[round_rows["Date"] < TEST_FROM].groupby("ScheduledRounds")["Outcome"]
          .value_counts(normalize=True).unstack().reindex(columns=ufc.OUTCOMES).fillna(0))

p_model = win_probability(winner_model, test)
how_a, how_b = how_it_ends(round_model, test), how_it_ends(round_model, ufc.swap(test))
p_elo = 1 / (1 + 10 ** (-test["D_Elo"].to_numpy() / 400))

def actual_cell_probability(tables):
    return np.array([t.loc[w, o] for t, w, o in zip(tables, test["Winner"], test["Outcome"])])

model_tables = [ufc.outcome_table(p, a, b, r) for p, a, b, r in zip(p_model, how_a, how_b, test["Rounds"].astype(int))]
elo_tables = [ufc.outcome_table(p, shares.loc[r], shares.loc[r], int(r)) for p, r in zip(p_elo, test["Rounds"])]
coin_tables = [ufc.outcome_table(0.5, shares.loc[r], shares.loc[r], int(r)) for r in test["Rounds"]]
# isolates the round model: the model's winner, but historical shares for when
model_winner_tables = [ufc.outcome_table(p, shares.loc[r], shares.loc[r], int(r)) for p, r in zip(p_model, test["Rounds"])]

a_won = (test["Winner"] == "A").to_numpy()
all_tables = [model_tables, model_winner_tables, elo_tables, coin_tables]
summary = pd.DataFrame({
    "Winner accuracy": [((p_model >= 0.5) == a_won).mean()] * 2 + [((p_elo >= 0.5) == a_won).mean(), np.nan],
    "Winner log loss": [log_loss(a_won, p_model)] * 2 + [log_loss(a_won, p_elo), log_loss(a_won, np.full(len(test), 0.5))],
    "Full-table log loss": [-np.log(actual_cell_probability(t)).mean() for t in all_tables],
    "Most likely cell happened": [np.mean([t.stack().idxmax() == (w, o) for t, w, o in zip(ts, test["Winner"], test["Outcome"])])
                                  for ts in all_tables],
}, index=["Both models", "Winner model + historical rounds", "Elo + historical rounds", "Coin flip + historical rounds"])
print(f"{len(test)} held-out fights from {test['Date'].min():%Y-%m-%d} to {test['Date'].max():%Y-%m-%d}")
summary.round(3)

802 held-out fights from 2025-01-11 to 2026-10-17


,Winner accuracy,Winner log loss,Full-table log loss,Most likely cell happened
Both models,0.636,0.657,1.865,0.322
Winner model + historical rounds,0.636,0.657,1.913,0.277
Elo + historical rounds,0.550,0.686,1.942,0.241
Coin flip + historical rounds,NaN,0.693,1.949,0.209


In [6]:
# Calibration: when the model gives its pick X%, does that fighter win about X% of the time?
pick = np.maximum(p_model, 1 - p_model)
pick_won = (p_model >= 0.5) == a_won
calibration = pd.DataFrame({"pick": pick, "won": pick_won}).groupby(
    pd.cut(pick, [0.5, 0.55, 0.6, 0.65, 0.7, 0.8, 1.0]), observed=True).agg(
    fights=("won", "size"), predicted=("pick", "mean"), actual=("won", "mean"))
calibration.round(3)

,fights,predicted,actual
"(0.5, 0.55]",166,0.526,0.542
"(0.55, 0.6]",164,0.577,0.622
"(0.6, 0.65]",154,0.626,0.656
"(0.65, 0.7]",136,0.676,0.691
"(0.7, 0.8]",161,0.742,0.689
"(0.8, 1.0]",21,0.813,0.571


## 4. Save the held-out predictions for the app's Results tab

In [7]:
names = {"R1": "in Round 1", "R2": "in Round 2", "R3": "in Round 3", "R4": "in Round 4", "R5": "in Round 5", "DEC": "by decision"}

def describe(fighter_a, fighter_b, side, outcome):
    return f"{fighter_a if side == 'A' else fighter_b} {names[outcome]}"

backtest = test[["FightID", "Date", "Event", "FighterA", "FighterB", "Rounds", "WeightClass"]].copy()
backtest["PWinA"], backtest["EloPWinA"] = p_model, p_elo
for side in ["A", "B"]:
    for o in ufc.OUTCOMES:
        backtest[f"{side}_{o}"] = [t.loc[side, o] if o in t.columns else np.nan for t in model_tables]
backtest["Actual"] = [describe(a, b, w, o) for a, b, w, o in zip(test["FighterA"], test["FighterB"], test["Winner"], test["Outcome"])]
backtest["TopCell"] = [describe(a, b, *t.stack().idxmax()) for a, b, t in zip(test["FighterA"], test["FighterB"], model_tables)]
backtest["PActual"] = actual_cell_probability(model_tables)
backtest["PickCorrect"] = (p_model >= 0.5) == a_won
backtest["EloPickCorrect"] = (p_elo >= 0.5) == a_won
backtest["LogLoss"] = -np.log(backtest["PActual"])
backtest["BaselineLogLoss"] = -np.log(actual_cell_probability(elo_tables))
PREDICTIONS.mkdir(parents=True, exist_ok=True)
backtest.to_csv(PREDICTIONS / f"UFC_backtest_v{VERSION}.csv", index=False)
backtest.head()

,FightID,Date,Event,FighterA,FighterB,Rounds,WeightClass,PWinA,EloPWinA,A_R1,...,B_R4,B_R5,B_DEC,Actual,TopCell,PActual,PickCorrect,EloPickCorrect,LogLoss,BaselineLogLoss
0,03c1865919daeaa4,2025-01-11,UFC Fight Night: Dern vs. Ribas 2,Austin Bashi,Christian Rodriguez,3.0,Featherweight,0.40660,0.452838,0.107586,...,NaN,NaN,0.326963,Christian Rodriguez by decision,Christian Rodriguez by decision,0.326963,True,True,1.117907,1.337069
1,192619174bd1ad77,2025-01-11,UFC Fight Night: Dern vs. Ribas 2,Joe Solecki,Nurullo Aliev,3.0,Lightweight,0.51285,0.524331,0.085347,...,NaN,NaN,0.308104,Nurullo Aliev by decision,Joe Solecki by decision,0.308104,False,False,1.177316,1.477093
2,32cc31b76b32ade1,2025-01-11,UFC Fight Night: Dern vs. Ribas 2,Punahele Soriano,Uros Medic,3.0,Welterweight,0.39420,0.449980,0.116419,...,NaN,NaN,0.260252,Punahele Soriano in Round 1,Uros Medic by decision,0.116419,False,False,2.150560,2.092695
3,54fa6e32330a8cca,2025-01-11,UFC Fight Night: Dern vs. Ribas 2,Thiago Moises,Trey Ogden,3.0,Lightweight,0.53970,0.509551,0.115160,...,NaN,NaN,0.288746,Thiago Moises by decision,Trey Ogden by decision,0.274734,True,True,1.291953,1.408285
4,58eda5a58a89f739,2025-01-11,UFC Fight Night: Dern vs. Ribas 2,Jacobe Smith,Preston Parsons,3.0,Welterweight,0.45085,0.514800,0.125664,...,NaN,NaN,0.289677,Jacobe Smith in Round 1,Preston Parsons by decision,0.125664,False,True,2.074140,1.958119


## 5. Final models for the app

Same model types and settings, refit on every fight including the held-out ones.

In [8]:
MODELS.mkdir(parents=True, exist_ok=True)
winner_exp.save_model(winner_exp.finalize_model(winner_model), str(MODELS / f"ufc_winner_model_v{VERSION}"))
round_exp.save_model(round_exp.finalize_model(round_model), str(MODELS / f"ufc_round_model_v{VERSION}"))

Transformation Pipeline and Model Successfully Saved


Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('label_encoding',
                  TransformerWrapperWithInverse(exclude=None, include=None,
                                                transformer=LabelEncoder())),
                 ('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['A_Fights', 'B_Fights', 'D_Fights',
                                              'A_KOWinRate', 'B_KOWinRate',
                                              'D_KOWinRate', 'A_SubWinRate',
                                              'B_SubWinRate', 'D_SubWinRate',
                                              'A_KOLossRate', 'B_KO...
                  TransformerWrapper(exclude=None, include=None,
                                     transformer=StandardScaler(copy=True,
                                                                with_mean=True,
                                                                with_std=Tru